# Multi-Factor Quant Trading System
Medium-term US equity strategy with backtesting

In [ ]:
# Setup: clone repo and install dependencies
!git clone https://github.com/yingwang/trade.git
%cd trade
!pip install -q -r requirements.txt

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

from quant.utils.config import load_config
from quant.strategy import MultiFactorStrategy
from quant.backtest.report import monthly_returns_table, risk_report

In [ ]:
# Load config and run backtest
config = load_config('config.yaml')
strategy = MultiFactorStrategy(config)
result = strategy.run_backtest()
print(result.summary())

In [ ]:
# Equity curve vs benchmark
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

ax = axes[0]
result.equity_curve.plot(ax=ax, label='Strategy', linewidth=1.5)
result.benchmark_curve.plot(ax=ax, label='SPY Benchmark', linewidth=1.5, alpha=0.7)
ax.set_title('Equity Curve'); ax.set_ylabel('$'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
dd = (result.equity_curve - result.equity_curve.cummax()) / result.equity_curve.cummax()
dd.plot(ax=ax, color='red', linewidth=1)
ax.fill_between(dd.index, dd.values, 0, alpha=0.3, color='red')
ax.set_title('Drawdown'); ax.grid(True, alpha=0.3)

ax = axes[2]
rolling_sharpe = (result.returns.rolling(63).mean() * 252) / (result.returns.rolling(63).std() * 252**0.5)
rolling_sharpe.plot(ax=ax, linewidth=1)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Rolling 3-Month Sharpe'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Monthly returns heatmap
import seaborn as sns

monthly = monthly_returns_table(result.equity_curve)
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(monthly, annot=True, fmt='.1%', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Monthly Returns'); plt.tight_layout(); plt.show()
monthly

In [ ]:
# Risk report
import pandas as pd
risk = risk_report(result.returns)
pd.Series(risk, name='Value')

In [ ]:
# Current live signals
signals = strategy.get_current_signal()
signals.head(20)